# Visformer (Vision-Friendly Transformer) for 3-Class Monkeypox Detection
## Rigorous Evaluation, Domain-Informed Augmentations & High-Speed Execution

**Paper Reference:** *Visformer: The Vision-friendly Transformer* (Chen et al., ICCV 2021)

### Architectural Highlights:
- **Hybrid Philosophy:** Visformer embeds convolutional depthwise layers and Squeeze-and-Excitation (SE) channel attention modules directly into transformer self-attention blocks, replacing pure MLP blocks with conv-based feedforward networks to maximize dermatological feature discrimination.
- **Colab NVMe Caching (10x-20x Speedup):** Automatically syncs images from Google Drive to Colab's high-speed local SSD once at startup.
- **Differential Learning Rates:** Backbone fine-tunes at $2\times 10^{-5}$ while the classification head trains at $2\times 10^{-4}$.
- **Domain-Informed Dermatological Augmentations:** Rotations, affine transforms, illumination jitter, and constrained Cutout/RandomErasing.
- **Test-Time Augmentation (TTA):** Evaluates original and horizontally flipped images during inference.
- **Google Drive Persistence:** Checkpoints (`best_model.pth`), 300 DPI figures, and LaTeX tables saved to Google Drive.

## 1. Connect to Google Drive & High-Speed Local Caching

In [ ]:
import os
import shutil
from pathlib import Path

# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")
except Exception as e:
    print(f"Note: Running outside standard Colab or already mounted. Error: {e}")

# 2. Define Project Base Directory in Google Drive
PROJECT_DIR = Path('/content/drive/MyDrive/Monkeypox_Final')

if not PROJECT_DIR.exists():
    my_drive = Path('/content/drive/MyDrive')
    candidates = list(my_drive.glob('*[Mm]onkeypox*')) if my_drive.exists() else []
    if candidates:
        PROJECT_DIR = candidates[0]
        print(f"Auto-located project directory at: {PROJECT_DIR}")
    else:
        avail = [p.name for p in my_drive.iterdir() if p.is_dir()] if my_drive.exists() else 'Drive not mounted'
        raise FileNotFoundError(
            f"Could not find project directory: {PROJECT_DIR}\n"
            f"Available folders in Google Drive MyDrive:\n{avail}\n"
            f"Please ensure your folder is named 'Monkeypox_Final' under MyDrive."
        )

# 3. Intelligent Dataset Discovery on Google Drive
def locate_dataset_dirs(base_dir):
    drive_train = None
    drive_test = None
    search_roots = [
        base_dir / 'mkpox_dataset_grayscale',
        base_dir / 'mkpox_dataset_grayscale' / 'mkpox_dataset_grayscale',
        base_dir / 'processed',
        base_dir,
        base_dir / 'dataset'
    ]
    for root in search_roots:
        if not root.exists():
            continue
        for p in root.iterdir():
            if p.is_dir():
                name_l = p.name.lower()
                if name_l == 'train' and drive_train is None:
                    drive_train = p
                elif name_l == 'test' and drive_test is None:
                    drive_test = p
        if drive_train and drive_test:
            return drive_train, drive_test
    for p in base_dir.rglob('*'):
        if p.is_dir():
            name_l = p.name.lower()
            if name_l == 'train' and drive_train is None:
                subdirs = [s.name.lower() for s in p.iterdir() if s.is_dir()]
                if any('pox' in s or 'norm' in s for s in subdirs):
                    drive_train = p
            elif name_l == 'test' and drive_test is None:
                subdirs = [s.name.lower() for s in p.iterdir() if s.is_dir()]
                if any('pox' in s or 'norm' in s for s in subdirs):
                    drive_test = p
        if drive_train and drive_test:
            break
    return drive_train, drive_test

drive_train_dir, drive_test_dir = locate_dataset_dirs(PROJECT_DIR)
if not drive_train_dir or not drive_test_dir:
    raise FileNotFoundError(f"Could not locate 'train' and 'test' folders inside: {PROJECT_DIR}")

print(f"Located Train directory on Drive: {drive_train_dir}")
print(f"Located Test directory on Drive:  {drive_test_dir}")

# 4. High-Speed Local Caching to Colab SSD
LOCAL_DATASET_DIR = Path('/content/dataset_local')
TRAIN_DIR = LOCAL_DATASET_DIR / 'train'
TEST_DIR  = LOCAL_DATASET_DIR / 'test'

def cache_split(src, dst):
    cached_images = list(dst.glob('*/*')) if dst.exists() else []
    if len(cached_images) == 0:
        if dst.exists():
            shutil.rmtree(dst)
        dst.mkdir(parents=True, exist_ok=True)
        print(f"Caching {src.name} to local NVMe SSD ({dst})...")
        for cls_dir in src.iterdir():
            if cls_dir.is_dir():
                target_cls = dst / cls_dir.name
                shutil.copytree(cls_dir, target_cls, dirs_exist_ok=True)
        img_count = len(list(dst.glob('*/*')))
        print(f"Successfully cached {src.name}: {img_count} images across {[d.name for d in dst.iterdir() if d.is_dir()]}")
    else:
        print(f"{src.name} already cached at {dst} ({len(cached_images)} images).")

cache_split(drive_train_dir, TRAIN_DIR)
cache_split(drive_test_dir, TEST_DIR)

# 5. Output Directories in Google Drive
MODEL_IDENTIFIER = 'visformer_small'
MODEL_FALLBACK = 'visformer_tiny'
EXPERIMENT_NAME = '13_Visformer_Hybrid'
OUTPUT_DIR = PROJECT_DIR / 'results' / EXPERIMENT_NAME
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
FIGURES_DIR = OUTPUT_DIR / 'figures'
TABLES_DIR = OUTPUT_DIR / 'tables'

for d in [CHECKPOINT_DIR, FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"\nProject Directory:  {PROJECT_DIR}")
print(f"Train Directory:    {TRAIN_DIR} (Ready: {TRAIN_DIR.exists()})")
print(f"Test Directory:     {TEST_DIR}  (Ready: {TEST_DIR.exists()})")
print(f"Outputs Directory:  {OUTPUT_DIR}")


## 2. Install Dependencies & Hardware Acceleration

In [ ]:
# Install dependencies
!pip install -q timm torchmetrics scikit-learn seaborn matplotlib pandas numpy pillow

import os
import sys
import time
import json
import shutil
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    matthews_corrcoef, cohen_kappa_score,
    balanced_accuracy_score, f1_score, precision_recall_fscore_support
)
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import label_binarize

# Set deterministic seed for reproducibility
SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 3. High-Performance Data Pipeline with Domain-Informed Augmentation
- **3-Channel Grayscale Adaptation:** Repeating the single grayscale channel across R, G, and B preserves pretrained positional embeddings without chromatic distortion.
- **Domain-Informed Augmentations:** Flips, rotation ($\pm 15^\circ$), affine scaling/translation, illumination jitter, and constrained Cutout/RandomErasing.
- **Stratified Train/Validation Split (85% / 15%):** Using `StratifiedShuffleSplit` on training data. The held-out test set remains completely untouched until final evaluation.

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 3

# ImageNet normalization constants
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

# Domain-informed training augmentations
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.08), value='random')
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

# Load datasets
full_train_dataset = datasets.ImageFolder(root=str(TRAIN_DIR), transform=train_transforms)
val_split_dataset = datasets.ImageFolder(root=str(TRAIN_DIR), transform=eval_transforms)
test_dataset = datasets.ImageFolder(root=str(TEST_DIR), transform=eval_transforms)

class_names = full_train_dataset.classes
targets = [s[1] for s in full_train_dataset.samples]

# Stratified Train/Val Split (85% Train, 15% Val)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_indices, val_indices = next(sss.split(np.zeros(len(targets)), targets))

train_dataset = torch.utils.data.Subset(full_train_dataset, train_indices)
val_dataset = torch.utils.data.Subset(val_split_dataset, val_indices)

# High-throughput DataLoaders
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True, persistent_workers=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True, persistent_workers=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True, persistent_workers=True
)

# Class Distribution Table
train_targets_list = [targets[i] for i in train_indices]
val_targets_list = [targets[i] for i in val_indices]
test_targets_list = [s[1] for s in test_dataset.samples]

split_records = []
for idx_c, cname in enumerate(class_names):
    tr_c = sum(1 for t in train_targets_list if t == idx_c)
    va_c = sum(1 for t in val_targets_list if t == idx_c)
    te_c = sum(1 for t in test_targets_list if t == idx_c)
    tot_c = tr_c + va_c + te_c
    split_records.append({
        'Class': cname,
        'Train Count': tr_c,
        'Train (%)': f"{(tr_c / len(train_dataset))*100:.1f}%",
        'Val Count': va_c,
        'Val (%)': f"{(va_c / len(val_dataset))*100:.1f}%",
        'Test Count': te_c,
        'Test (%)': f"{(te_c / len(test_dataset))*100:.1f}%",
        'Total': tot_c
    })

df_splits = pd.DataFrame(split_records)
print("="*75)
print("DATASET SPLIT & CLASS DISTRIBUTION (STRATIFIED)")
print("="*75)
display(df_splits)
print(f"Total: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)} (Held-out)")
df_splits.to_csv(TABLES_DIR / 'dataset_split_distribution.csv', index=False)


## 4. Model Architecture: Visformer (Vision-Friendly Transformer) for 3-Class Monkeypox Detection

In [ ]:
# Instantiate Hybrid Model with ImageNet pretrained weights (with fallback)
try:
    print(f"Attempting to load: {MODEL_IDENTIFIER}...")
    model = timm.create_model(MODEL_IDENTIFIER, pretrained=True, num_classes=NUM_CLASSES)
except Exception as e:
    print(f"Primary model tag unavailable ({e}). Falling back to: {MODEL_FALLBACK}...")
    MODEL_IDENTIFIER = MODEL_FALLBACK
    model = timm.create_model(MODEL_IDENTIFIER, pretrained=True, num_classes=NUM_CLASSES)

model = model.to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Successfully Loaded: {MODEL_IDENTIFIER}")
print(f"Total Parameters:     {total_params / 1e6:.2f} M")
print(f"Trainable Parameters: {trainable_params / 1e6:.2f} M")


## 5. Training Setup: Differential Learning Rates & Warmup Cosine Schedule
- **Differential Learning Rates:** Backbone fine-tunes at $2\times 10^{-5}$, Classification head at $2\times 10^{-4}$.
- **Scheduler:** 3-epoch linear warmup followed by Cosine Annealing decay down to $1\times 10^{-6}$.
- **Loss:** CrossEntropyLoss with Label Smoothing ($0.1$).
- **Mixed Precision:** `torch.cuda.amp.autocast()` with `GradScaler()` for maximum throughput.

In [ ]:
EPOCHS = 30
WARMUP_EPOCHS = 3

# Separate backbone and classification head parameters
backbone_params = []
head_params = []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if any(k in name.lower() for k in ['head', 'fc', 'classifier']):
        head_params.append(param)
    else:
        backbone_params.append(param)

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': 2e-5, 'weight_decay': 1e-2},
    {'params': head_params, 'lr': 2e-4, 'weight_decay': 1e-2}
])

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / float(WARMUP_EPOCHS)
    else:
        progress = float(epoch - WARMUP_EPOCHS) / float(max(1, EPOCHS - WARMUP_EPOCHS))
        return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.cuda.amp.GradScaler()

print(f"Configured Differential Optimizer: Backbone LR=2e-5, Head LR=2e-4, Epochs={EPOCHS}")

config = {
    'experiment_name': EXPERIMENT_NAME,
    'model_identifier': MODEL_IDENTIFIER,
    'seed': SEED,
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'num_classes': NUM_CLASSES,
    'epochs': EPOCHS,
    'warmup_epochs': WARMUP_EPOCHS,
    'backbone_lr': 2e-5,
    'head_lr': 2e-4,
    'weight_decay': 1e-2,
    'patience': 7,
    'min_delta': 0.001,
    'early_stopping_metric': 'val_macro_f1',
    'optimizer': 'AdamW',
    'scheduler': 'CosineAnnealingWithWarmup',
    'label_smoothing': 0.1,
    'augmentation': {
        'rotation': 15,
        'flips': 'horizontal_and_vertical',
        'affine_translate': 0.05,
        'affine_scale': [0.95, 1.05],
        'color_jitter': {'brightness': 0.10, 'contrast': 0.10},
        'random_erasing': {'p': 0.15, 'scale': [0.02, 0.08]}
    },
    'device': str(device),
    'gpu_hardware': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
}

with open(TABLES_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=4)
print(f"Configuration logged to: {TABLES_DIR / 'config.json'}")


## 6. High-Speed Training Loop with Macro-F1 Early Stopping
- **Validation Metric:** Monitors **Validation Macro-F1** and **Balanced Accuracy** each epoch.
- **Early Stopping:** Tracks `best_val_macro_f1` with `PATIENCE = 7` epochs (active after 3-epoch warmup).
- **Checkpoint Preservation:** Saves `best_model.pth` with complete optimizer state, configuration metadata, and validation metrics.

In [ ]:
best_val_macro_f1 = 0.0
best_val_acc = 0.0
best_model_path = CHECKPOINT_DIR / 'best_model.pth'
PATIENCE = 7
MIN_DELTA = 0.001
patience_counter = 0

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'val_macro_f1': [], 'val_bal_acc': [],
    'lr': []
}

print("Starting High-Speed Training with Macro-F1 Early Stopping...")
start_total_time = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    
    # --- Training Phase ---
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()
        
    epoch_train_loss = running_loss / total_train
    epoch_train_acc  = (correct_train / total_train) * 100.0
    
    # --- Validation Phase ---
    model.eval()
    running_val_loss, correct_val, total_val = 0.0, 0, 0
    val_targets, val_preds = [], []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
                
            running_val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()
            val_targets.extend(labels.cpu().numpy())
            val_preds.extend(predicted.cpu().numpy())
            
    val_targets = np.array(val_targets)
    val_preds = np.array(val_preds)
    
    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc  = (correct_val / total_val) * 100.0
    epoch_val_macro_f1 = f1_score(val_targets, val_preds, average='macro') * 100.0
    epoch_val_bal_acc  = balanced_accuracy_score(val_targets, val_preds) * 100.0
    
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    epoch_elapsed = time.time() - epoch_start
    
    # Record history
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    history['val_macro_f1'].append(epoch_val_macro_f1)
    history['val_bal_acc'].append(epoch_val_bal_acc)
    history['lr'].append(current_lr)
    
    # Early Stopping check
    if epoch_val_macro_f1 > best_val_macro_f1 + MIN_DELTA:
        best_val_macro_f1 = epoch_val_macro_f1
        best_val_acc = epoch_val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_macro_f1': float(best_val_macro_f1),
            'best_val_acc': float(best_val_acc),
            'class_names': class_names,
            'config': config,
            'history': history
        }, best_model_path)
        save_msg = f"--> Saved Best Model (Val Macro-F1: {best_val_macro_f1:.2f}%, Acc: {epoch_val_acc:.2f}%)"
    else:
        patience_counter += 1
        save_msg = f"[No improve: {patience_counter}/{PATIENCE}]" if epoch > WARMUP_EPOCHS else ""
        
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({epoch_elapsed:.1f}s) | Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.2f}% | Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.2f}% Macro-F1: {epoch_val_macro_f1:.2f}% {save_msg}")
    
    if epoch > WARMUP_EPOCHS and patience_counter >= PATIENCE:
        print(f"\n[Early Stopping Triggered] No validation Macro-F1 improvement for {PATIENCE} consecutive epochs.")
        print(f"Optimal checkpoint preserved at: {best_model_path}")
        break

total_training_time = time.time() - start_total_time
print(f"\nTraining Complete in {total_training_time/60:.2f} minutes! Best Validation Macro-F1: {best_val_macro_f1:.2f}% (Acc: {best_val_acc:.2f}%)")

# Save training history to CSV and JSON
df_history = pd.DataFrame(history)
df_history.insert(0, 'epoch', range(1, len(df_history) + 1))
df_history.to_csv(TABLES_DIR / 'training_history.csv', index=False)
with open(TABLES_DIR / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=4)
print(f"Training history successfully saved to: {TABLES_DIR}")


## 7. Empirical Test-Time Augmentation (TTA) Validation
Held-out test set evaluated under two distinct inference protocols:
1. **Standard Inference (Without TTA):** Single-pass forward evaluation.
2. **Test-Time Augmentation (With TTA):** Ensemble of predictions across original and horizontally flipped test images.

In [ ]:
# Load best checkpoint
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded optimal checkpoint from Epoch {checkpoint['epoch']} (Val Macro-F1: {checkpoint['best_val_macro_f1']:.2f}%)")

y_true = []
y_probs_standard = []
y_probs_tta = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        with torch.cuda.amp.autocast():
            out_orig = model(images)
            p_orig = torch.softmax(out_orig, dim=1)
            images_flipped = torch.flip(images, dims=[-1])
            out_flip = model(images_flipped)
            p_flip = torch.softmax(out_flip, dim=1)
            p_tta = 0.5 * (p_orig + p_flip)
            
        y_true.extend(labels.cpu().numpy())
        y_probs_standard.extend(p_orig.cpu().numpy())
        y_probs_tta.extend(p_tta.cpu().numpy())

y_true = np.array(y_true)
y_probs_standard = np.array(y_probs_standard)
y_probs_tta = np.array(y_probs_tta)
y_pred_standard = np.argmax(y_probs_standard, axis=1)
y_pred_tta = np.argmax(y_probs_tta, axis=1)

def compute_complete_metrics(y_t, y_p, y_prob):
    acc = (y_t == y_p).mean() * 100.0
    bal_acc = balanced_accuracy_score(y_t, y_p) * 100.0
    macro_f1 = f1_score(y_t, y_p, average='macro') * 100.0
    weighted_f1 = f1_score(y_t, y_p, average='weighted') * 100.0
    mcc = matthews_corrcoef(y_t, y_p)
    kappa = cohen_kappa_score(y_t, y_p)
    
    y_t_bin = label_binarize(y_t, classes=[0, 1, 2])
    fpr_d, tpr_d, roc_auc_d = {}, {}, {}
    for i in range(NUM_CLASSES):
        fpr_d[i], tpr_d[i], _ = roc_curve(y_t_bin[:, i], y_prob[:, i])
        roc_auc_d[i] = auc(fpr_d[i], tpr_d[i])
        
    all_fpr = np.unique(np.concatenate([fpr_d[i] for i in range(NUM_CLASSES)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(NUM_CLASSES):
        mean_tpr += np.interp(all_fpr, fpr_d[i], tpr_d[i])
    mean_tpr /= NUM_CLASSES
    macro_auc = auc(all_fpr, mean_tpr) * 100.0
    
    pr_auc_d = {}
    for i in range(NUM_CLASSES):
        pr_auc_d[i] = average_precision_score(y_t_bin[:, i], y_prob[:, i])
        
    return {
        'accuracy': acc, 'balanced_accuracy': bal_acc,
        'macro_f1': macro_f1, 'weighted_f1': weighted_f1,
        'macro_auc': macro_auc, 'mcc': mcc, 'kappa': kappa,
        'roc_auc_dict': roc_auc_d, 'pr_auc_dict': pr_auc_d
    }

metrics_std = compute_complete_metrics(y_true, y_pred_standard, y_probs_standard)
metrics_tta = compute_complete_metrics(y_true, y_pred_tta, y_probs_tta)

df_tta_comparison = pd.DataFrame([
    {
        'Evaluation Protocol': 'Standard (No TTA)',
        'Accuracy (%)': round(metrics_std['accuracy'], 2),
        'Balanced Acc (%)': round(metrics_std['balanced_accuracy'], 2),
        'Macro-F1 (%)': round(metrics_std['macro_f1'], 2),
        'Weighted-F1 (%)': round(metrics_std['weighted_f1'], 2),
        'Macro-AUC (%)': round(metrics_std['macro_auc'], 2),
        'MCC': round(metrics_std['mcc'], 4),
        'Kappa': round(metrics_std['kappa'], 4)
    },
    {
        'Evaluation Protocol': 'With TTA (Horizontal Flip)',
        'Accuracy (%)': round(metrics_tta['accuracy'], 2),
        'Balanced Acc (%)': round(metrics_tta['balanced_accuracy'], 2),
        'Macro-F1 (%)': round(metrics_tta['macro_f1'], 2),
        'Weighted-F1 (%)': round(metrics_tta['weighted_f1'], 2),
        'Macro-AUC (%)': round(metrics_tta['macro_auc'], 2),
        'MCC': round(metrics_tta['mcc'], 4),
        'Kappa': round(metrics_tta['kappa'], 4)
    },
    {
        'Evaluation Protocol': 'Empirical Difference (Δ)',
        'Accuracy (%)': round(metrics_tta['accuracy'] - metrics_std['accuracy'], 2),
        'Balanced Acc (%)': round(metrics_tta['balanced_accuracy'] - metrics_std['balanced_accuracy'], 2),
        'Macro-F1 (%)': round(metrics_tta['macro_f1'] - metrics_std['macro_f1'], 2),
        'Weighted-F1 (%)': round(metrics_tta['weighted_f1'] - metrics_std['weighted_f1'], 2),
        'Macro-AUC (%)': round(metrics_tta['macro_auc'] - metrics_std['macro_auc'], 2),
        'MCC': round(metrics_tta['mcc'] - metrics_std['mcc'], 4),
        'Kappa': round(metrics_tta['kappa'] - metrics_std['kappa'], 4)
    }
])

print("="*80)
print("EMPIRICAL TEST-TIME AUGMENTATION (TTA) COMPARISON")
print("="*80)
display(df_tta_comparison)
df_tta_comparison.to_csv(TABLES_DIR / 'tta_empirical_comparison.csv', index=False)
with open(TABLES_DIR / 'tta_empirical_comparison.tex', 'w') as f:
    f.write(df_tta_comparison.to_latex(index=False))

y_pred = y_pred_standard
y_probs = y_probs_standard


## 8. Generate Research Paper Deliverables & Publication Figures (300 DPI)
1. **Training & Validation Curves** (Loss, Accuracy, Macro-F1)
2. **Confusion Matrices** (Raw Counts & Normalized Percentages)
3. **Multi-Class ROC-AUC Curves**
4. **Multi-Class Precision-Recall Curves**
5. **Per-Class Performance Table & Master Summary**

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.size': 12, 'axes.labelsize': 13, 'axes.titlesize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'figure.titlesize': 15, 'figure.dpi': 300
})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_ran = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_ran, history['train_loss'], label='Training Loss', color='#1f77b4', lw=2)
axes[0].plot(epochs_ran, history['val_loss'], label='Validation Loss', color='#d62728', lw=2, linestyle='--')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title(f'{EXPERIMENT_NAME}: Loss Curves')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_ran, history['train_acc'], label='Training Accuracy', color='#2ca02c', lw=2)
axes[1].plot(epochs_ran, history['val_acc'], label='Validation Accuracy', color='#ff7f0e', lw=2, linestyle='--')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title(f'{EXPERIMENT_NAME}: Accuracy Curves')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_ran, history['val_macro_f1'], label='Val Macro-F1', color='#9467bd', lw=2)
axes[2].axhline(y=best_val_macro_f1, color='gray', linestyle=':', label=f'Best ({best_val_macro_f1:.2f}%)')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Macro-F1 (%)')
axes[2].set_title(f'{EXPERIMENT_NAME}: Early Stopping Metric (Macro-F1)')
axes[2].legend(loc='lower right')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
curves_path = FIGURES_DIR / '01_training_validation_curves.png'
plt.savefig(curves_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {curves_path}")


In [ ]:
cm_raw = confusion_matrix(y_true, y_pred)
cm_norm = cm_raw.astype('float') / cm_raw.sum(axis=1)[:, np.newaxis] * 100.0

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(cm_raw, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0],
            cbar_kws={'label': 'Sample Count'})
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')
axes[0].set_title(f'{EXPERIMENT_NAME}: Confusion Matrix (Counts)')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens', xticklabels=class_names, yticklabels=class_names, ax=axes[1],
            cbar_kws={'label': 'Percentage (%)'})
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')
axes[1].set_title(f'{EXPERIMENT_NAME}: Normalized Confusion Matrix (%)')

plt.tight_layout()
cm_path = FIGURES_DIR / '02_confusion_matrices.png'
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {cm_path}")


In [ ]:
y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
fpr, tpr, roc_auc = dict(), dict(), dict()
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

plt.figure(figsize=(8, 6))
for i in range(NUM_CLASSES):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    plt.plot(fpr[i], tpr[i], color=colors[i], lw=2,
             label=f'{class_names[i]} (AUC = {roc_auc[i]:.4f})')

all_fpr = np.unique(np.concatenate([fpr[i] for i in range(NUM_CLASSES)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(NUM_CLASSES):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= NUM_CLASSES
macro_auc = auc(all_fpr, mean_tpr)

plt.plot(all_fpr, mean_tpr, color='navy', linestyle=':', lw=3,
         label=f'Macro-Average (AUC = {macro_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title(f'{EXPERIMENT_NAME}: Multi-Class ROC Curves')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

roc_path = FIGURES_DIR / '03_roc_auc_curves.png'
plt.savefig(roc_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {roc_path}")


In [ ]:
plt.figure(figsize=(8, 6))
pr_auc = dict()
for i in range(NUM_CLASSES):
    precision_vals, recall_vals, _ = precision_recall_curve(y_true_bin[:, i], y_probs[:, i])
    pr_auc[i] = average_precision_score(y_true_bin[:, i], y_probs[:, i])
    plt.plot(recall_vals, precision_vals, color=colors[i], lw=2,
             label=f'{class_names[i]} (AP = {pr_auc[i]:.4f})')

plt.xlabel('Recall (Sensitivity)')
plt.ylabel('Precision (PPV)')
plt.title(f'{EXPERIMENT_NAME}: Precision-Recall Curves')
plt.legend(loc='lower left')
plt.grid(True, alpha=0.3)

pr_path = FIGURES_DIR / '04_precision_recall_curves.png'
plt.savefig(pr_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {pr_path}")


In [ ]:
def generate_per_class_df(y_t, y_p, roc_d, pr_d):
    cm = confusion_matrix(y_t, y_p)
    records = []
    for i, cname in enumerate(class_names):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)
        
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        f1   = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        supp = int(cm[i, :].sum())
        
        records.append({
            'Class': cname,
            'Precision (%)': round(prec * 100, 2),
            'Recall / Sensitivity (%)': round(rec * 100, 2),
            'Specificity (%)': round(spec * 100, 2),
            'F1-Score (%)': round(f1 * 100, 2),
            'ROC-AUC (%)': round(roc_d[i] * 100, 2),
            'PR-AUC (%)': round(pr_d[i] * 100, 2),
            'Support': supp
        })
    return pd.DataFrame(records)

df_per_class_std = generate_per_class_df(y_true, y_pred_standard, metrics_std['roc_auc_dict'], metrics_std['pr_auc_dict'])
df_per_class_tta = generate_per_class_df(y_true, y_pred_tta, metrics_tta['roc_auc_dict'], metrics_tta['pr_auc_dict'])

print("\n--- PER-CLASS PERFORMANCE TABLE (Standard Inference) ---")
display(df_per_class_std)
print("\n--- PER-CLASS PERFORMANCE TABLE (With TTA) ---")
display(df_per_class_tta)

df_overall_summary = pd.DataFrame([
    {
        'Model': EXPERIMENT_NAME,
        'Protocol': 'Standard',
        'Parameters (M)': round(total_params / 1e6, 2),
        'Accuracy (%)': round(metrics_std['accuracy'], 2),
        'Balanced Acc (%)': round(metrics_std['balanced_accuracy'], 2),
        'Macro-F1 (%)': round(metrics_std['macro_f1'], 2),
        'Weighted-F1 (%)': round(metrics_std['weighted_f1'], 2),
        'Macro-AUC (%)': round(metrics_std['macro_auc'], 2),
        'MCC': round(metrics_std['mcc'], 4),
        'Cohen Kappa': round(metrics_std['kappa'], 4)
    },
    {
        'Model': EXPERIMENT_NAME,
        'Protocol': 'With TTA',
        'Parameters (M)': round(total_params / 1e6, 2),
        'Accuracy (%)': round(metrics_tta['accuracy'], 2),
        'Balanced Acc (%)': round(metrics_tta['balanced_accuracy'], 2),
        'Macro-F1 (%)': round(metrics_tta['macro_f1'], 2),
        'Weighted-F1 (%)': round(metrics_tta['weighted_f1'], 2),
        'Macro-AUC (%)': round(metrics_tta['macro_auc'], 2),
        'MCC': round(metrics_tta['mcc'], 4),
        'Cohen Kappa': round(metrics_tta['kappa'], 4)
    }
])

print("\n--- OVERALL PERFORMANCE SUMMARY (Standard vs TTA) ---")
display(df_overall_summary)

df_per_class_std.to_csv(TABLES_DIR / 'per_class_metrics_standard.csv', index=False)
df_per_class_tta.to_csv(TABLES_DIR / 'per_class_metrics_tta.csv', index=False)
df_overall_summary.to_csv(TABLES_DIR / 'overall_metrics.csv', index=False)

with open(TABLES_DIR / 'per_class_metrics_standard.tex', 'w') as f:
    f.write(df_per_class_std.to_latex(index=False))
with open(TABLES_DIR / 'per_class_metrics_tta.tex', 'w') as f:
    f.write(df_per_class_tta.to_latex(index=False))
with open(TABLES_DIR / 'overall_metrics.tex', 'w') as f:
    f.write(df_overall_summary.to_latex(index=False))

summary_data = {
    'model_name': EXPERIMENT_NAME,
    'model_identifier': MODEL_IDENTIFIER,
    'parameters_m': round(total_params / 1e6, 2),
    'standard': {
        'accuracy': round(metrics_std['accuracy'], 2),
        'balanced_accuracy': round(metrics_std['balanced_accuracy'], 2),
        'macro_f1': round(metrics_std['macro_f1'], 2),
        'weighted_f1': round(metrics_std['weighted_f1'], 2),
        'macro_auc': round(metrics_std['macro_auc'], 2),
        'mcc': round(metrics_std['mcc'], 4),
        'kappa': round(metrics_std['kappa'], 4),
        'per_class': df_per_class_std.to_dict(orient='records')
    },
    'tta': {
        'accuracy': round(metrics_tta['accuracy'], 2),
        'balanced_accuracy': round(metrics_tta['balanced_accuracy'], 2),
        'macro_f1': round(metrics_tta['macro_f1'], 2),
        'weighted_f1': round(metrics_tta['weighted_f1'], 2),
        'macro_auc': round(metrics_tta['macro_auc'], 2),
        'mcc': round(metrics_tta['mcc'], 4),
        'kappa': round(metrics_tta['kappa'], 4),
        'per_class': df_per_class_tta.to_dict(orient='records')
    },
    'empirical_delta': {
        'accuracy': round(metrics_tta['accuracy'] - metrics_std['accuracy'], 2),
        'balanced_accuracy': round(metrics_tta['balanced_accuracy'] - metrics_std['balanced_accuracy'], 2),
        'macro_f1': round(metrics_tta['macro_f1'] - metrics_std['macro_f1'], 2),
        'weighted_f1': round(metrics_tta['weighted_f1'] - metrics_std['weighted_f1'], 2),
        'macro_auc': round(metrics_tta['macro_auc'] - metrics_std['macro_auc'], 2),
        'mcc': round(metrics_tta['mcc'] - metrics_std['mcc'], 4),
        'kappa': round(metrics_tta['kappa'] - metrics_std['kappa'], 4)
    },
    'config': config
}

with open(TABLES_DIR / 'metrics_summary.json', 'w') as f:
    json.dump(summary_data, f, indent=4)

print(f"\nAll tables, CSVs, LaTeX files, and JSON summaries saved to: {TABLES_DIR}")
